<style>
.markdown-rendered p, 
.markdown-rendered li {
    line-height: 2.0 !important;
    font-size: 3.0em !important;
}
</style>


# 📘 반도체 패키징 역설계 핵심 기법 & 알고리즘 상세 해설서 (v2)

이 문서는 AI 기반 반도체 패키징 역설계 파이프라인(Step 1 ~ Step 5)에 사용된 데이터 사이언스 및 전산역학(CAE) 기법들을 입문자의 눈높이에 맞추어 상세한 비유와 함께 정리한 백서입니다.

> 🆕 표시 = v2에서 새로 추가된 기법

---

## [Step 1] 대리 모델 학습 및 데이터 증강

### 1. 절댓값 Max Peak 추출 (`abs().idxmax()`)
* **1. 기법 소개:** 시간에 따라 위아래로 요동치는 파형에서, 방향 무관하게 가장 파괴적인 충격 순간을 찾아내는 전처리 기법입니다.
* **2. 주요 특징:** 절댓값으로 크기를 비교해 위치를 찾은 뒤, 원본 부호(+/-)를 살려서 가져옵니다. 팽창(+)인지 수축(-)인지의 물리적 방향성을 보존합니다.
* **3. 작동 원리:** `[10, -30, 20]` → 절댓값 `[10, 30, 20]` → 최대 위치(index 1) → 원본값 `-30` 추출.
* **4. 프로젝트 도입 이유:** 냉각 시 치명적인 압축 응력(-)을 단순 max()가 무시하는 것을 방지합니다.

### 2. 피어슨 상관계수 (Pearson Correlation Coefficient)
* **1. 기법 소개:** 두 데이터 흐름의 선형 관계를 -1~1 사이 수치로 보여주는 통계 기법입니다.
* **2. 주요 특징:** 1=정비례, -1=반비례, 0=무관.
* **3. 작동 원리:** 공분산을 각 표준편차로 나누어 정규화합니다.
* **4. 프로젝트 도입 이유:** 딥러닝 학습 전 히트맵으로 P↔Y 인과관계를 팩트 체크하기 위함입니다.

### 3. GPR (가우시안 프로세스 회귀)
* **1. 기법 소개:** 데이터 점들을 통과할 수 있는 '무수히 많은 부드러운 곡선들의 확률 분포'를 통째로 예측하는 베이지안 ML 알고리즘입니다.
* **2. 주요 특징:** 예측값(μ)과 함께 불확실성(σ)을 반드시 출력합니다.
* **3. 작동 원리:** 관측된 데이터를 기반으로 미지의 영역은 주변 거리를 참고해 확률 지도를 업데이트합니다.
* **4. 프로젝트 도입 이유:** 트리 모델의 양 끝단 뿔(Clipping) 현상을 극복하고, ~900개 데이터에 최적인 부드러운 대리 모델을 위해 선정했습니다.

### 4. ARD 커널 (Automatic Relevance Determination)
* **1. 기법 소개:** GPR이 각 설계 변수의 중요도를 스스로 깨닫게 만드는 커널입니다.
* **2. 주요 특징:** Length Scale을 P1~P6 각각에 독립 부여합니다.
* **3. 작동 원리:** 둔감한 변수는 눈금을 늘려 무시, 예민한 변수는 좁혀서 미세 변화를 캐치합니다.
* **4. 프로젝트 도입 이유:** 6개 두께 변수 중 진짜 범인을 AI가 수학적으로 분리하기 위함입니다.

### 5. 라틴 하이퍼큐브 샘플링 (LHS)
* **1. 기법 소개:** 다차원 공간에 점들을 겹치지 않고 골고루 흩뿌리는 고급 난수 생성 알고리즘입니다.
* **2. 주요 특징:** 단순 랜덤의 뭉침 현상을 수학적으로 원천 차단합니다.
* **3. 작동 원리:** 각 차원을 균등 격자로 쪼갠 뒤, 어떤 줄에도 데이터가 딱 1개씩만 배치됩니다.
* **4. 프로젝트 도입 이유:** 10만 개 가상 조합의 6차원 공간을 사각지대 없이 커버하기 위함입니다.

---

## [Step 2] 은닉 제약조건 분류기 (Gatekeeper)

### 6. 랜덤 포레스트 분류기 (Random Forest Classifier)
* **1. 기법 소개:** 수백 개의 의사결정 나무를 모아 다수결 투표로 결론을 내리는 앙상블 ML 기법입니다.
* **2. 주요 특징:** 개별 나무의 편견이 다수결로 상쇄되어 집단 지성이 발휘됩니다.
* **3. 작동 원리:** 100그루의 나무가 각자 Safe/Fail 판별 후, 많은 쪽으로 최종 판정합니다.
* **4. 프로젝트 도입 이유:** 물리적 파괴 조합의 불규칙한 경계선을 방어하는 방패 역할입니다.

### 7. 클래스 불균형 해소 (`class_weight='balanced'`)
* **1. 기법 소개:** 데이터 비율 쏠림 시 AI가 다수파에 편승하는 꼼수를 차단하는 교정 기법입니다.
* **2. 주요 특징:** 소수 데이터에 막대한 가중치를 부여하여 기울어진 운동장을 평평하게 만듭니다.
* **3. 작동 원리:** 불량 데이터를 틀리면 정상 데이터 대비 훨씬 더 큰 벌점을 부여합니다.
* **4. 프로젝트 도입 이유:** 25%에 불과한 파탄 조합을 하나라도 놓치지 않기 위함입니다.

---

## [Step 3] 파레토 프론티어 타겟 곡선 추출

### 8. 파레토 비지배 정렬 (Pareto Non-dominated Sorting)
* **1. 기법 소개:** 상충하는 두 목적 동시 달성 시, 누구에게도 꿀리지 않는 절대적 1등 설계들의 집합을 솎아내는 서열 필터입니다.
* **2. 주요 특징:** 두 목적 함수를 타협 없는 독립 잣대로 유지하며 최상위 Frontier를 찾습니다.
* **3. 작동 원리:** A가 B보다 두 목적 모두 우수하면 B를 도태시키고, 아무에게도 지배당하지 않는 개체만 Frontier 0으로 묶습니다.
* **4. 프로젝트 도입 이유:** 유토피아 타겟을 원본 데이터에서 발굴하기 위함입니다.

---

## [Step 4] 오토인코더 잠재 매핑 역설계

### 9. 🆕 시계열 리샘플링 (`np.interp` 선형 보간)
* **1. 기법 소개:** 서로 다른 길이(612~631 타임스텝)의 시계열을 **공통 시간축(600포인트)으로 통일**하는 전처리 기법입니다.
* **2. 주요 특징:** 단순 절단/제로패딩과 달리 원본 시간축의 물리적 의미(0~300초)를 보존한 채 균등 간격으로 재배치합니다.
* **3. 작동 원리:** 새 시간축의 각 지점에서 원본의 좌우 이웃 값을 직선으로 연결해 중간 값을 추정합니다.
* **4. 프로젝트 도입 이유:** 오토인코더 입력은 길이가 동일해야 하는데, 시뮬레이션마다 타임스텝 수가 달라 3D 텐서 생성이 불가했기 때문입니다.

### 10. 🆕 사비츠키-골레이 필터 (Savitzky-Golay Filter)
* **1. 기법 소개:** 고주파 노이즈(메쉬 진동)를 제거하면서 **가열-유지-냉각 경계의 급격한 변화(계단형 전이)를 보존**하는 신호 처리 필터입니다. 이동평균의 상위 호환입니다.
* **2. 주요 특징:** 각 윈도우 내에서 다항식을 독립 피팅하므로 경계를 뭉개지 않습니다. 윈도우 크기(11포인트 ≈ 5.5초)가 스텝 전이(수십 초)보다 짧아 경계를 통과시킵니다.
* **3. 작동 원리:** 11개 연속 포인트 윈도우를 한 칸씩 밀며 3차 다항식을 최소제곱법으로 피팅하고, 중앙값을 새 데이터로 채택합니다.
* **4. 프로젝트 도입 이유:** FEA 메쉬 재배치의 수치적 진동이 오토인코더 학습을 방해하되, 열 사이클링 3단계 경계를 오염시키지 않는 필터가 필요했습니다.

### 11. 🆕 채널별 StandardScaler (독립 정규화)
* **1. 기법 소개:** 7개 채널의 스케일 차이(WarpMax: ~0.1 vs SEQV: ~40)를 해소하기 위해 **채널마다 독립적으로 평균=0, 표준편차=1로 정규화**합니다.
* **2. 주요 특징:** 7개 채널 각각에 별도 scaler 객체를 생성하며, 추론 시 역변환을 위해 보관합니다.
* **3. 작동 원리:** 채널 i의 전체 데이터에서 $(x - \mu_i) / \sigma_i$로 변환합니다.
* **4. 프로젝트 도입 이유:** 정규화 없이 학습하면 절댓값이 큰 채널에만 Loss가 집중되어 작은 채널의 복원이 무시됩니다.

### 12. 1D-CNN 오토인코더 (Autoencoder)
* **1. 기법 소개:** 시계열을 병목(Bottleneck)으로 압축→복원하는 훈련으로 데이터의 핵심 요약본(잠재 벡터)을 터득하는 딥러닝 기술입니다.
* **2. 주요 특징:** 비지도 학습. 1D-CNN이 시간 흐름 파동 패턴 스캔에 특화되어 있습니다.
* **3. 작동 원리:** 4,200개(600×7) 시계열을 합성곱으로 32개 잠재 벡터로 압축하고, 디코더가 이를 복원합니다.
* **4. 프로젝트 도입 이유:** 고차원→6개 두께 직접 역추적이 불안정하므로 32차원 압축 공간을 거치는 2단계 우회 전략입니다.

### 13. 🆕 Residual Block (잔차 연결, ResNet)
* **1. 기법 소개:** 각 블록에 **입력을 그대로 통과시키는 지름길(Skip Path)**을 추가하여 원본과의 '차이(잔차)'만 학습하도록 유도합니다. 2015년 ResNet 논문에서 시작되었습니다.
* **2. 주요 특징:** 기울기 소실 문제를 근본 해결합니다. 수십~수백 층도 안정 학습 가능합니다.
* **3. 작동 원리:** $H(x) = F(x) + x$. 미세한 보정분 $F(x)$만 배우면 되므로 학습 난이도가 획기적으로 낮아집니다.
* **4. 프로젝트 도입 이유:** 계단형 전이에서 CNN 디코더의 깁스 현상을 완화합니다. 경계는 스킵 경로로 보존되고, 네트워크는 잔차만 학습합니다.

### 14. 🆕 U-Net Skip Connection (인코더-디코더 직접 연결)
* **1. 기법 소개:** 인코더 각 레벨의 중간 피처맵을 디코더 대응 레벨에 **직접 이어붙여(Concatenate)** 전달합니다. 의료 영상 분할용 U-Net에서 유래했습니다.
* **2. 주요 특징:** 인코더가 압축하며 잃어버린 세밀한 시간축 정보(경계 위치, 전이 속도)를 디코더가 직접 참조합니다.
* **3. 작동 원리:** Encoder 각 레벨을 Decoder 대응 레벨에 채널 축으로 concat한 뒤 ResBlock으로 융합합니다.
* **4. 프로젝트 도입 이유:** 32차원 잠재 벡터만으로는 600×7 시계열의 계단 경계 위치가 손실됩니다. 다해상도 정보를 직접 주입하여 복원 정밀도를 향상시킵니다.

### 15. 🆕 Upsample + Conv1d (체커보드 아티팩트 제거)
* **1. 기법 소개:** 디코더 업샘플링 시 ConvTranspose 대신 **Nearest Neighbor 업샘플링 + 일반 Conv1d**를 조합합니다.
* **2. 주요 특징:** ConvTranspose의 커널 stride 불일치로 발생하는 체커보드 아티팩트를 구조적으로 원천 차단합니다.
* **3. 작동 원리:** `nn.Upsample(nearest)`로 파라미터 없이 2배 복제 후, `nn.Conv1d`로 스무딩합니다.
* **4. 프로젝트 도입 이유:** ConvTranspose 디코더의 고주파 진동이 잠재 벡터 품질을 오염시켰습니다. 교체 후 Val Loss 40% 감소.

### 16. 🆕 Smooth L1 Loss (Huber Loss 변형)
* **1. 기법 소개:** 오차가 작으면 MSE처럼, 크면 MAE처럼 동작하는 **하이브리드 손실 함수**입니다.
* **2. 주요 특징:** MSE의 큰 오차 제곱 페널티가 계단 경계에서 오버슈트(깁스 현상)를 유발하는 것을 억제합니다.
* **3. 작동 원리:** 오차가 β(=0.1) 이하면 MSE, 이상이면 L1으로 계산합니다.
* **4. 프로젝트 도입 이유:** MSE만 사용 시 디코더가 계단 경계에서 출렁거리는 깁스 현상이 관찰되어 TV Loss와 함께 적용했습니다.

### 17. 🆕 Total Variation Loss (TV Loss, 평탄화 규제)
* **1. 기법 소개:** 복원 시계열에서 인접 타임스텝 간 급격한 변동에 페널티를 부여하여 **파형을 매끄럽게 다림질하는 보조 손실 함수**입니다.
* **2. 주요 특징:** 메인 복원 Loss와 독립적 보조 규제(Regularizer). 가중치(λ=0.05)로 강도 조절.
* **3. 작동 원리:** $TV = \frac{1}{N}\sum|\hat{x}_{t+1} - \hat{x}_t|$를 Loss에 더하여 지글거림을 억제합니다.
* **4. 프로젝트 도입 이유:** Smooth L1과 ResBlock으로도 유지 구간의 미세 진동이 잔존하여, TV Loss로 잔여 진동까지 제거했습니다.

### 18. 🆕 지도형 오토인코더 (Supervised Autoencoder)
* **1. 기법 소개:** 잠재 벡터 $z$가 **설계변수(P1~P6) 정보를 반드시 담도록 강제하는 보조 Loss를 추가**한 비지도+지도 하이브리드 학습 전략입니다.
* **2. 주요 특징:** U-Net Skip이 강력할수록 디코더가 $z$를 무시하는 '정보 우회(Information Bypass)' 현상이 발생합니다. P-예측 보조 Loss가 이를 해결합니다.
* **3. 작동 원리:** 총 Loss = 복원(SmoothL1) + TV + $\lambda_{sup}$ × MSE($z$ → P 예측). 보조 MLP가 $z$에서 P를 예측하고, 틀리면 $z$에 벌점이 가해집니다.
* **4. 프로젝트 도입 이유:** ResNet+U-Net으로 복원은 완벽해졌으나 역매핑(z→P)이 무너졌습니다. '게으른 잠재 벡터' 문제를 P-예측 Loss(λ=1.0)로 해결했습니다.

### 19. 다층 퍼셉트론 (MLP) 역매핑
* **1. 기법 소개:** 뇌 신경망을 모방한 범용 딥러닝 모델로, 입출력 간 비선형 관계를 스스로 찾아냅니다.
* **2. 주요 특징:** 노드와 층을 여러 겹 쌓아 복잡한 패턴을 학습합니다.
* **3. 작동 원리:** 오차를 역전파하여 가중치를 수만 번 미세 조절합니다.
* **4. 프로젝트 도입 이유:** 오토인코더가 뽑은 32개 압축 암호를 P1~P6로 번역하는 역설계 통역기입니다.

---

## [Step 5] NSGA-II 기반 통합 강건 최적화

### 20. NSGA-II (다목적 유전 알고리즘)
* **1. 기법 소개:** 다윈의 진화론을 모방하여 수백 세대에 걸쳐 최적 변수 조합을 찾는 메타 휴리스틱 알고리즘입니다.
* **2. 주요 특징:** 파레토 비지배 정렬을 내부 탑재, 세대마다 우월한 유전자만 선별합니다.
* **3. 작동 원리:** Step 4 초안을 부모로 삼아 교배/돌연변이→엘리트 선별을 100세대 반복합니다.
* **4. 프로젝트 도입 이유:** 역매핑 초안의 정밀도를 ±10% 범위에서 미세조정하기 위함입니다.

### 21. 강건 최적화 ($\mu + 2\sigma < Limit$)
* **1. 기법 소개:** 대리 모델의 오차 범위(σ)까지 포함시켜 **최악의 상황에서도 파괴되지 않도록 보수적으로 설계**하는 엔지니어링 철학입니다.
* **2. 주요 특징:** "최악의 경우에도 살아남을 수 있는가?"를 핵심 기준으로 삼습니다.
* **3. 작동 원리:** GPR 예측값 μ에 2σ(95% 신뢰구간)를 더한 값이 파괴 한계선을 넘는지 검사합니다.
* **4. 프로젝트 도입 이유:** AI 예측값만 맹신하면 Ansys 검증에서 터질 수 있어, 2σ 마진으로 실패율을 0%에 방어합니다.

### 22. Feasibility Rule (`pymoo` `G` Matrix)
* **1. 기법 소개:** 제약 위반 개체를 무작정 실격하지 않고, **위반량을 비교하여 진화 방향을 알려주는 벌점 방식**입니다.
* **2. 주요 특징:** +999,999 상수 페널티와 달리, 위반량을 연속 실수로 추적합니다.
* **3. 작동 원리:** 만족하면 음수(-), 위반하면 초과량 비례 양수(+) 점수. G값이 0 이하로 떨어지도록 유도합니다.
* **4. 프로젝트 도입 이유:** 묻지마 대형 벌점은 진화 방향을 잃게 만들어, 부드러운 수렴을 위해 도입했습니다.

### 23. Knee Point (최적 밸런스 점 추출)
* **1. 기법 소개:** 파레토 곡선에서 가장 볼록한 **무릎(Knee) 위치**의 궁극적 타협점을 콕 집어내는 선택 기법입니다.
* **2. 주요 특징:** 양 끝단의 큰 희생 없이 두 목표 모두에서 이득을 취하는 '최고 가성비 구간'입니다.
* **3. 작동 원리:** 두 목적함수를 [0,1] 정규화 후, 원점 (0,0)과의 유클리드 거리가 최소인 점을 Rank 1위로 지정합니다.
* **4. 프로젝트 도입 이유:** 극단 편향 방지, 휨과 박리를 최적 밸런스로 방어하는 '황금 1개'를 자동 추천합니다.

# 차후 개선 방안


1. 2D 모델링 및 형상의 단순화, 재질 고정 등의 한계

본 프로젝트에서는 2D 모델링을 사용하여 기판 등 파트의 형상의 복잡성을 단순화 하였습니다.
또한, 팬에 의한 냉각 같은 유동 해석 문제도 간략화를 통해 생략했습니다.
실제 현실에 가까운 결과를 원할 경우 실제 3d 형상에 유동해석도 포함해야 합니다만
이 경우 실제 논문들에서 사용하는 슈퍼 컴퓨터를 사용하지 않으면 몇 개월을 24시간 풀로 돌려도 충분한 데이터를 얻을수 없습니다.

2. 패러미터 지정 

이번 실험에서 선정한 두께 패러미터는 실제로는 파운드리나 칩 설계사가 이미 고정해놓은 상수로 변경이 어렵습니다.

warpage의 경우 핵심 요인이 CTE(열팽창계수 차이), 온도변화량, 탄성계수, 두께 이기 때문에 EMC의 배합(CTE), 공정온도 (이번 실험에서는 120 -> 22도로 고정), si-bridge 두께나 substrtate의 내부 구조 등이 더 좋은 패러이터
(reference. Artificial Intelligence-Based Warpage Prediction Model for Accelerating Thermo-Mechanical Simulation in Advanced Packaging,  2025 IEEE 75th Electronic Components and Technology Conference (ECTC).)

두께의 경우 소숫점 단위의 최적 수치를 도출하더라도 절대 이와 정확한 수치를 적용하기 어렵습니다. 현실적으로 가공오차는 필연적이며 정밀가공을 하더라도 단가나 납기의 문제가 발생합니다.

3. 대리모델의 방향성

본래 기존의 대리모델의 개념에 맞게 구현하려면 모든 time step에 대응하는 시계열 데이터를 예측하고 구현하는 대리모델을 만들어야 하지만 이는 논문 수준의 매우 어려운 작업이기에 이번에는 절대값의 최대치만 예측하는 대리모델을 구현했습니다.

하지만 높은 정확도의 시계열 데이터를 예측하는 대리모델을 구현할 경우 step4 에서 학습시킬때 더 많은 데이터를 기반으로 더 정확한 결과를 도출할 수 있습니다.

다만 본 프로젝트와 같이 단순한 패러미터의 경우 과한 기능일수 있습니다.

4. 샘플링 방법 

본 프로젝트에서는 난수 생성 샘플링 기법을 Qusi-Random Search에 포함하는 Latin Hypercube Sampling(LHS)을 사용했습니다.
다음에는 adpative experimentation에 해당하는 bayesian optimization을 사용하여 더 뛰어난 효율성을 기대해볼수 있습니다.